<a href="https://colab.research.google.com/github/dianasouzza/tech-challenge-olist/blob/main/notebooks/02_consultas_sql_duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rodando consultas SQL direto sobre os arquivos CSV no Google Colab usando o DuckDB.

In [1]:
#Libs

#Importar o DuckDB para trabalhar com SQL

!pip install -q duckdb
import duckdb
import pandas as pd

# Visuais

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Consulta SQL para mensalização de pedidos e ticket médio

# receita_com_frete é na verdade o valor total pago pelos clientes (RB + Frete), também chamado de GMV.

query = """
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS ano_mes,
        COUNT(DISTINCT o.order_id) AS total_pedidos,
        ROUND(SUM(i.price)) AS receita_total,
        ROUND(SUM(i.freight_value), 2) AS frete_total,
        ROUND(SUM(i.price + i.freight_value), 2) AS receita_com_frete,
        ROUND(SUM(i.price + i.freight_value) / COUNT(DISTINCT o.order_id)) AS ticket_medio
    FROM 'olist_orders_dataset.csv' o
    JOIN 'olist_order_items_dataset.csv' i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    AND o.order_purchase_timestamp >= '2017-01-01'
    AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY 1
    ORDER BY 1
"""

# Executa a query via DuckDB e transforma em DataFrame
df_mensal = duckdb.query(query).df()

# Exibe o resultado formatado na tela
display(df_mensal)

,ano_mes,total_pedidos,receita_total,frete_total,receita_com_frete,ticket_medio
0,2017-01,750,111798.0,15684.01,127482.37,170.0
1,2017-02,1653,234223.0,37015.92,271239.32,164.0
2,2017-03,2546,359199.0,55132.10,414330.95,163.0
3,2017-04,2303,340670.0,50142.72,390812.40,170.0
4,2017-05,3546,489338.0,77513.15,566851.40,160.0
5,2017-06,3135,421923.0,68127.00,490050.37,156.0
6,2017-07,3872,481605.0,84694.56,566299.08,146.0
7,2017-08,4193,554700.0,91132.66,645832.36,154.0
8,2017-09,4150,607400.0,93677.82,701077.49,169.0
9,2017-10,4478,648248.0,102869.36,751117.01,168.0


In [3]:
df_mensal.describe()

,total_pedidos,receita_total,frete_total,receita_com_frete,ticket_medio
count,20.00000,20.000000,20.000000,2.000000e+01,20.000000
mean,4810.55000,659051.350000,109604.644000,7.686560e+05,160.600000
std,2016.19155,272819.171792,48229.676079,3.206784e+05,7.625442
min,750.00000,111798.000000,15684.010000,1.274824e+05,146.000000
25%,3443.25000,466684.500000,75166.612500,5.472369e+05,154.750000
50%,4995.50000,687140.500000,109957.230000,7.970977e+05,161.500000
75%,6603.50000,882126.000000,153906.952500,1.040327e+06,167.000000
max,7289.00000,987765.000000,167241.990000,1.153364e+06,170.000000


No período de Jan/17 a Ago/18 a media mensal é de:

 ~4.800 pedidos/mes

 ~659k receita/mes

 ~R$138 ticket médio mensal

 No mes de Jul/17 o ticket médio foi o menor chegando a 124 e fev/18 126

 O mês com receita mais alta foi nov/17, com ~988k e 7.289 pedidos, influenciado, provavelmente, pelo mes sazonal da Black Friday, mantendo um TKM saudável de 136, muito próximo a média mensal de 138. Porém nao temos nov/16 e nem nov/18 para comparar.

In [4]:
# --- GRÁFICO 1: Receita vs Volume de Pedidos (Eixo Duplo) ---
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Traço da Receita (Eixo Y esquerdo)
fig.add_trace(
    go.Scatter(
        x=df_mensal['ano_mes'], y=df_mensal['receita_com_frete'],
        mode='lines+markers', name='Receita Total (R$)',
        line=dict(color='#1f77b4', width=2.5)
    ),
    secondary_y=False
)

# Traço do Total de Pedidos (Eixo Y direito - secundário)
fig.add_trace(
    go.Scatter(
        x=df_mensal['ano_mes'], y=df_mensal['total_pedidos'],
        mode='lines+markers', name='Total de Pedidos',
        line=dict(color='#ff7f0e', width=2.5, dash='dash')
    ),
    secondary_y=True
)

# Layout do Gráfico 1
fig.update_layout(
    title_text='<b>Evolução da Operação Olist: Receita vs Volume de Pedidos</b>',
    title_x=0.5,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.update_xaxes(title_text='Ano-Mês')
fig.update_yaxes(title_text='<b>Receita Total (R$)</b>', secondary_y=False, rangemode='tozero')
fig.update_yaxes(title_text='<b>Total de Pedidos</b>', secondary_y=True, rangemode='tozero')

fig.show()


# --- GRÁFICO 2: Ticket Médio Mensal (com exibição dos valores nos pontos) ---
fig1 = px.line(
    df_mensal,
    x='ano_mes',
    y='ticket_medio',
    text='ticket_medio',  # Adiciona o valor do ticket médio como texto
    markers=True,
    title='<b>Evolução Interativa do Ticket Médio Mensal (Olist)</b>',
    labels={'ano_mes': 'Ano-Mês', 'ticket_medio': 'Ticket Médio (R$)'}
)

# Ajusta a posição do texto para ficar centralizado acima de cada ponto do gráfico
fig1.update_traces(textposition='top center')

fig1.update_layout(
    title_x=0.5,
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    hovermode='x unified',
    template='plotly_white'
)

fig1.update_yaxes(rangemode='tozero')

fig1.show()

# Exploração da tabela de reviews

In [5]:
# Consulta SQL para explorar a base de reviews

# pct_nota_baixa é a porcentagem de pedidos do mês com nota 1 ou 2.
# Em jan/2017, 12,0% dos pedidos entregues tiveram nota baixa.
# Na média 12% de nota baixa, porém alguns meses sobem para 16,5%, chegando a 21% de notas baixas
query = """
WITH nota_pedido AS (
    SELECT order_id, AVG(review_score) AS nota
    FROM 'olist_order_reviews_dataset.csv'
    GROUP BY order_id
)
SELECT
    strftime(o.order_purchase_timestamp, '%Y-%m') AS ano_mes,
    ROUND(AVG(n.nota), 2) AS score_medio,
    COUNT(*) AS total_pedidos,
    ROUND(100.0 * AVG(CASE WHEN n.nota <= 2 THEN 1 ELSE 0 END), 1) AS pct_nota_baixa
FROM 'olist_orders_dataset.csv' o
JOIN nota_pedido n ON o.order_id = n.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
GROUP BY 1
ORDER BY 1
"""

# Executa a query via DuckDB e transforma em DataFrame
df_reviews = duckdb.query(query).df()

# Exibe o resultado formatado na tela
display(df_reviews)

,ano_mes,score_medio,total_pedidos,pct_nota_baixa
0,2017-01,4.20,741,12.0
1,2017-02,4.20,1643,11.1
2,2017-03,4.19,2527,11.2
3,2017-04,4.14,2290,12.8
4,2017-05,4.24,3518,10.0
5,2017-06,4.22,3111,10.6
6,2017-07,4.26,3842,9.9
7,2017-08,4.31,4165,9.4
8,2017-09,4.27,4118,10.2
9,2017-10,4.21,4446,11.0


In [6]:
df_reviews.describe()

,score_medio,total_pedidos,pct_nota_baixa
count,20.000000,20.000000,20.000000
mean,4.175500,4778.400000,12.245000
std,0.140206,2004.323522,3.279357
min,3.810000,741.000000,9.400000
25%,4.132500,3416.250000,10.000000
50%,4.210000,4953.500000,11.050000
75%,4.262500,6560.750000,13.025000
max,4.320000,7238.000000,21.100000


In [7]:
# --- GRÁFICO: Evolução da Nota Média Mensal ---
import plotly.express as px

fig_score = px.line(
    df_reviews,
    x='ano_mes',
    y='score_medio',
    text=df_reviews['score_medio'].round(2),  # Arredonda para 2 casas decimais no texto do ponto
    markers=True,
    title='<b>Evolução da Nota Média Mensal de Avaliação (Olist)</b>',
    labels={'ano_mes': 'Ano-Mês', 'score_medio': 'Nota Média (1 a 5)'}
)

# Ajusta a posição do texto para ficar centralizado acima de cada ponto do gráfico
fig_score.update_traces(textposition='top center')

fig_score.update_layout(
    title_x=0.5,
    title_font_size=16,
    hovermode='x unified',
    template='plotly_white'
)

# Como as notas vão de 1 a 5, é interessante fixar o eixo Y nesse intervalo para dar perspectiva
fig_score.update_yaxes(range=[1, 5.2])

fig_score.show()